In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from services.abrir_plantilla_doris import abrir_plantilla_doris
from services.manager_variables_excel_datos_despliegue import *
from services.manager_variables_excel_datos_campania import *
from services.manager_variables_excel_documento import *
from services.doc_103.constructor_diccionario_datos_documento import *
from services.word_template_writer import *
from services.word_template_writer._table_helpers import procesar_titulos_y_referencias, activar_actualizacion_de_campos

from services.guardar_documento import guardar_documento
from services.doc_103.manager_porcentajes import *

from configs.manager_doc_config import configuracion_documentos
from services.doc_103.copiar_archivos import copiar_archivos


In [ ]:
from configs.configuracion_103 import doc_config


In [ ]:
config = configuracion_documentos(doc_config)


In [ ]:
## Del despliegue
# Cargar datos de despliegue
ruta_al_excel_de_despliegue = config.get_ruta_al_excel_de_despliegue_de_sondas()
nombre_de_hoja_despliegue = config.get_hoja_del_excel_de_despliegue_de_sondas()
df_datos_despliegue = get_datos_de_despliegue(ruta_al_excel_de_despliegue, nombre_de_hoja_despliegue)
df_datos_despliegue_filtrado = filtrar_df_datos_despliegue_por_orden_de_servicio(df_datos_despliegue = df_datos_despliegue, orden_de_servicio=config.get_orden_de_servicio(), columna="OS_103")



In [ ]:
seriales_de_sondas = get_seriales_de_sondas(df_datos_despliegue_filtrado)

In [ ]:
## De la campania
ruta_al_excel_de_campanias = config.get_ruta_al_excel_de_campanias()
nombre_de_hoja_campanias = config.get_hoja_del_excel_de_campanias()
df_campanias = get_df_datos_campanias(ruta_al_excel_de_campanias, nombre_de_hoja_campanias)
df_campanias_filtrado = filtrar_datos_de_campanias(df_datos_campanias=df_campanias, seriales_de_sondas=seriales_de_sondas)
    

In [ ]:
# Cargar datos del excel para el documento
ruta_al_excel_para_crear_documento = config.get_ruta_al_excel_para_crear_documento()
hoja_del_excel_para_crear_documento = config.get_hoja_del_excel_para_crear_documento()
df_documento = get_df_datos_documento(ruta_al_excel_para_crear_documento, hoja_del_excel_para_crear_documento)

In [ ]:
# Informacion de la los datos procesados
ruta_al_excel_de_porcentajes = os.path.join(config.get_ruta_a_carpeta_de_las_figuras(),"porcentajes_de_las_sondas.xlsx")
df_porcentajes = get_df_porcentajes(ruta_al_excel_de_porcentajes)

In [ ]:
doc = abrir_plantilla_doris(config.get_ruta_a_la_plantilla_de_word())

In [ ]:
config.get_ruta_a_carpeta_de_plantilla()

In [ ]:
# copiar necesarios de la carpeta de despliegue
copiar_archivos(
    carpeta_de_origen = config.get_ruta_a_la_carpeta_de_despliegue(),
    carpeta_de_destino = config.get_ruta_a_carpeta_de_las_figuras(),
    lista_de_archivos = ["mapa_de_despliegue.png"]
)

# copiar necesarios de la carpeta de plantilla
copiar_archivos(
    carpeta_de_origen = config.get_ruta_a_carpeta_de_plantilla(),
    carpeta_de_destino = config.get_ruta_a_carpeta_de_las_figuras(),
    lista_de_archivos = ["esquema_componentes_de_sonda.jpg"]
)


In [ ]:
dict_demo = construir_diccionario_agregar_figuras(
    df_datos_documento = df_documento, 
    ruta_a_carpeta_de_figuras = config.get_ruta_a_carpeta_de_las_figuras(),
    df_datos_despliegue = df_datos_despliegue, 
    df_datos_campanias = df_campanias_filtrado,
    df_porcentajes = df_porcentajes
)

In [ ]:
construir_diccionario_de_portada(
    seriales_de_sondas = seriales_de_sondas,
    df_datos_despliegue=df_datos_despliegue_filtrado,
    diccionario_de_reemplazos=dict_demo,
    orden_de_servicio=config.get_orden_de_servicio()
)   

In [ ]:
construir_diccionario_de_datos_documento(
    df_datos_despliegue = df_datos_despliegue_filtrado,
    df_datos_campanias = df_campanias_filtrado,
    df_porcentajes = df_porcentajes,
    diccionario_de_reemplazos = dict_demo,
    config = config,
    seriales_de_sondas = seriales_de_sondas
)

In [ ]:
construir_diccionario_de_reemplazos_para_tablas(
    df_datos_despliegue = df_datos_despliegue_filtrado,
    df_porcentajes = df_porcentajes,
    df_datos_campanias = df_campanias_filtrado,
    diccionario_de_reemplazos = dict_demo,
    seriales_de_sondas = seriales_de_sondas,
    doc = doc
)

In [ ]:
reemplazar_texto_en_cuadros_de_texto(doc=doc, diccionario_de_reemplazos=dict_demo)
reemplazar_texto_en_plantilla(doc=doc,diccionario_de_reemplazos=dict_demo)
insertar_destino_en_marcador(doc=doc,marcador="<<destino_entregable_101>>",bookmark="referencia_entregable_101",numero="1")
insertar_figuras_en_plantilla(doc=doc, diccionario_de_reemplazos=dict_demo)
insertar_referencias_cruzadas_en_plantilla(doc=doc, diccionario_de_reemplazos=dict_demo)
insertar_referencia_en_marcador(doc=doc,marcador="<<ref_entregable_101>>",bookmark="referencia_entregable_101")
insertar_documento_externo_en_plantilla(doc=doc, diccionario_de_reemplazos=dict_demo)
rellenar_tablas_en_plantilla(doc=doc,diccionario_de_reemplazos=dict_demo)
reemplazar_variables_en_tablas(doc=doc, diccionario_de_reemplazos=dict_demo)

procesar_titulos_y_referencias(doc=doc,diccionario_de_reemplazos=dict_demo)
activar_actualizacion_de_campos(doc=doc)

In [ ]:
ruta_a_la_carpeta_de_guardado = config.get_ruta_a_carpeta_de_guardado_del_documento()

# Calcular fecha para el nombre del documento: primer día del mes siguiente a la vigencia
fecha_inicio_vigencia = get_fecha_inicio_vigencia(df_datos_despliegue_filtrado)
if not isinstance(fecha_inicio_vigencia, pd.Timestamp) or pd.isna(fecha_inicio_vigencia):
    raise RuntimeError("Fecha de inicio de vigencia inválida para generar nombre de archivo")
# primer día del mes siguiente
fecha_para_nombre = (fecha_inicio_vigencia + pd.DateOffset(months=1)).replace(day=1)
date_part = fecha_para_nombre.strftime("%Y%m%d")

nombre_de_archivo = f"{config.nombre_de_salida_del_documento}_{date_part}_00.docx"

guardar_documento(doc, ruta_a_la_carpeta_de_guardado=ruta_a_la_carpeta_de_guardado, nombre_archivo=nombre_de_archivo)